# Rainfall Prediction Classifier

A binary classification project that predicts whether it will rain, built on real-world
weather station data from the Australian Bureau of Meteorology. The notebook walks through
a complete applied machine learning workflow: cleaning raw observational data, engineering
features while avoiding data leakage, building a preprocessing + model pipeline, tuning
hyperparameters with grid search cross-validation, and evaluating results with an eye on
class imbalance rather than accuracy alone.

**Author:** Lewis
**Stack:** Python, pandas, scikit-learn, matplotlib, seaborn

## What this project covers

- Cleaning and filtering a real-world weather dataset
- Reasoning about data leakage and reframing the prediction target to avoid it
- Feature engineering (deriving a `Season` feature from raw dates)
- Building a `ColumnTransformer` + `Pipeline` for numeric and categorical preprocessing
- Hyperparameter tuning with `GridSearchCV` and stratified cross-validation
- Evaluating a classifier on an imbalanced target using precision, recall and the
  confusion matrix, not just accuracy
- Comparing a Random Forest classifier against Logistic Regression
- Extracting and visualizing feature importances


## About the dataset

The original source of the data is the Australian Government's Bureau of Meteorology
([www.bom.gov.au/climate/dwo](http://www.bom.gov.au/climate/dwo/)). This project uses a
version of the dataset downloaded from
[Kaggle](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package), with
column definitions sourced from the
[Bureau of Meteorology's station data](http://www.bom.gov.au/climate/dwo/IDCJDW0000.shtml).

The dataset contains daily weather observations from 2008 to 2017:

| Field         | Description                                           | Unit            | Type   |
| :------------ | :---------------------------------------------------- | :-------------- | :----- |
| Date          | Date of the observation (YYYY-MM-DD)                  | Date            | object |
| Location      | Location of the observation                           | Location        | object |
| MinTemp       | Minimum temperature                                   | Celsius         | float  |
| MaxTemp       | Maximum temperature                                   | Celsius         | float  |
| Rainfall      | Amount of rainfall                                    | Millimeters     | float  |
| Evaporation   | Amount of evaporation                                 | Millimeters     | float  |
| Sunshine      | Amount of bright sunshine                             | hours           | float  |
| WindGustDir   | Direction of the strongest gust                       | Compass Points  | object |
| WindGustSpeed | Speed of the strongest gust                           | Kilometers/Hour | object |
| WindDir9am    | Wind direction averaged over 10 minutes prior to 9am  | Compass Points  | object |
| WindDir3pm    | Wind direction averaged over 10 minutes prior to 3pm  | Compass Points  | object |
| WindSpeed9am  | Wind speed averaged over 10 minutes prior to 9am      | Kilometers/Hour | float  |
| WindSpeed3pm  | Wind speed averaged over 10 minutes prior to 3pm      | Kilometers/Hour | float  |
| Humidity9am   | Humidity at 9am                                       | Percent         | float  |
| Humidity3pm   | Humidity at 3pm                                       | Percent         | float  |
| Pressure9am   | Atmospheric pressure reduced to mean sea level at 9am | Hectopascal     | float  |
| Pressure3pm   | Atmospheric pressure reduced to mean sea level at 3pm | Hectopascal     | float  |
| Cloud9am      | Fraction of the sky obscured by cloud at 9am          | Eighths         | float  |
| Cloud3pm      | Fraction of the sky obscured by cloud at 3pm          | Eighths         | float  |
| Temp9am       | Temperature at 9am                                    | Celsius         | float  |
| Temp3pm       | Temperature at 3pm                                    | Celsius         | float  |
| RainToday     | Whether there was at least 1mm of rain today          | Yes/No          | object |
| RainTomorrow  | Whether there is at least 1mm of rain tomorrow         | Yes/No          | object |


## Setup

Install and import the required libraries.

In [ ]:
!pip install numpy pandas matplotlib scikit-learn seaborn -q

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## Load the data

In [ ]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/_0eYOqji3unP1tDNKWZMjg/weatherAUS-2.csv"
df = pd.read_csv(url)
df.head()

In [ ]:
df.count()

Sunshine and cloud cover look like potentially useful features, but they're missing far too many values to impute reliably, so it's simplest to drop rows with any missing values and see how much data is left.

In [ ]:
df = df.dropna()
df.info()

There are still around 56k observations left after dropping missing values, which should be plenty to work with.

In [ ]:
df.columns

## Data leakage considerations

Looking at the column descriptions, several of them are measured over the *entire* day
(`MaxTemp`, `Evaporation`, `WindGustDir`, `WindGustSpeed`, ...). If the goal is to predict
whether it will rain **tomorrow**, using today's full-day summary statistics as inputs is a
problem: those values genuinely aren't known until the day is already over, so a model
"predicting tomorrow" using them wouldn't actually be usable in practice — it would be
leaking future information.

**Features that would be inefficient/unusable for predicting tomorrow's rain include:**
`MaxTemp`, `Evaporation`, `WindGustDir`, `WindGustSpeed`, `Sunshine`, and the 3pm readings —
anything that summarizes or is only known at the end of the day. Features like `Humidity9am`
or `Temp9am` are fine, since they're tied to a specific, already-passed point in the day.

To sidestep this cleanly, the target is reframed: instead of predicting *tomorrow's* rain
from *today's* full-day data, the model predicts **today's** rain using yesterday's data —
this makes every feature legitimately available at prediction time (useful for something
like "should I bike to work today?"). The rain columns are renamed to reflect this shift.

In [ ]:
df = df.rename(columns={'RainToday': 'RainYesterday',
                         'RainTomorrow': 'RainToday'})

## Location selection

Weather predictability varies a lot by location across Australia, and using every location
at once would need a much more complex model to capture local patterns. Melbourne,
Melbourne Airport (18km away) and Watsonia (15km away) are close enough to be treated as one
region, so the analysis is narrowed to just those three, while keeping `Location` as a
categorical feature to capture any remaining local variation.

In [ ]:
df = df[df.Location.isin(['Melbourne', 'MelbourneAirport', 'Watsonia'])]
df.info()

That leaves 7,557 records — enough to build a reasonably good model (more locations or a
longer time window could always be added later).

## Feature engineering: mapping dates to seasons

Rain is seasonal, so instead of using the raw `Date` column directly, it's more useful to
derive a `Season` feature from it and drop the original date.

In [ ]:
def date_to_season(date):
    month = date.month
    if month in (12, 1, 2):
        return 'Summer'
    elif month in (3, 4, 5):
        return 'Autumn'
    elif month in (6, 7, 8):
        return 'Winter'
    elif month in (9, 10, 11):
        return 'Spring'

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Season'] = df['Date'].apply(date_to_season)
df = df.drop(columns='Date')
df.head()

## Define the feature and target sets

With a good set of features in place, the next step is to split the data into features
(`X`) and the target (`y`) — but first it's worth checking how balanced the target classes
are, since that shapes everything downstream (metric choice, stratification, model
interpretation).

In [ ]:
X = df.drop(columns='RainToday')
y = df['RainToday']

### Class balance

In [ ]:
y.value_counts()

**Observations on class balance:**

- It rains on roughly 24% of days in the Melbourne area (1,791 out of 7,557 observations).
- A model that always predicted "No rain" would already be about 76% accurate without
  learning anything meaningful — this naive baseline is the bar any real model needs to
  clear.
- The dataset is unbalanced, at roughly a 76/24 "No"/"Yes" split.
- Because of this, the next steps are: split the data with stratified sampling (to preserve
  this ratio in train/test), and evaluate any model with metrics beyond plain accuracy —
  precision, recall, F1-score, and the confusion matrix — since accuracy alone is
  misleading here.

## Split into training and test sets

Using `stratify=y` keeps the 76/24 class ratio consistent between the training and test
sets, rather than risking an even more skewed split by chance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## Preprocessing pipeline

Numeric and categorical features need different treatment: numeric features are scaled,
and categorical features are one-hot encoded. Column names are detected automatically by
dtype rather than hardcoded, so the pipeline stays reusable if the feature set changes.

In [ ]:
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

In [ ]:
# Scale the numeric features
numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])

# One-hot encode the categorical features
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## Model pipeline: Random Forest

The preprocessor is combined with a `RandomForestClassifier` into a single pipeline, so
preprocessing and modeling happen together on every fit/predict call — including inside
cross-validation.

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

### Hyperparameter grid for the grid search

In [ ]:
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

### Grid search cross-validation

`StratifiedKFold` keeps the class ratio consistent across every cross-validation fold, the
same way `stratify=y` did for the train/test split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='accuracy', verbose=2)
grid_search.fit(X_train, y_train)

In [ ]:
print("Best parameters found:", grid_search.best_params_)
print("Best cross-validation score: {:.2f}".format(grid_search.best_score_))

## Evaluate on the held-out test set

In [ ]:
test_score = grid_search.score(X_test, y_test)
print("Test set score: {:.2f}".format(test_score))

That's a reasonably accurate classifier on the surface — roughly 84% of predictions are "
"correct. But given the class imbalance uncovered earlier, accuracy alone doesn't tell the "
"whole story, so it's worth digging into the confusion matrix and classification report.

In [ ]:
y_pred = grid_search.predict(X_test)

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=grid_search.classes_)
disp.plot(cmap='Blues')
plt.title("Random Forest — Confusion Matrix")
plt.show()

**Is this a good predictor of rainfall?**

The true positive rate (recall on "Yes") is TP / (TP + FN) = 178 / (178 + 180) ≈ 50% — the
model catches only about half of the actual rainy days. That's a meaningful weakness: a
model whose overall accuracy looks solid (~84%) but whose recall on the class that actually
matters is close to a coin flip isn't yet a reliable rainfall predictor.

## Feature importances

To map feature importances back to their original column names, the one-hot encoded
categorical feature names need to be reconstructed from the fitted preprocessor — numeric
feature names are unaffected, since preprocessing didn't rename them.

In [ ]:
feature_importances = grid_search.best_estimator_['classifier'].feature_importances_

In [ ]:
# Combine numeric and categorical feature names
feature_names = numeric_features + list(
    grid_search.best_estimator_['preprocessor']
    .named_transformers_['cat']
    .named_steps['onehot']
    .get_feature_names_out(categorical_features)
)

feature_importances = grid_search.best_estimator_['classifier'].feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

N = 20
top_features = importance_df.head(N)

plt.figure(figsize=(10, 6))
plt.barh(top_features['Feature'], top_features['Importance'], color='skyblue')
plt.gca().invert_yaxis()
plt.title(f'Top {N} Most Important Features in Predicting Rain')
plt.xlabel('Importance Score')
plt.show()

`Humidity3pm` is the single most important feature, with an importance score of about 0.12 — well ahead of pressure and sunshine, which form the next tier.

## Comparing against a second model: Logistic Regression

In practice you'd want to try several model types before settling on one. Reusing the same
`Pipeline` and `GridSearchCV` objects, the classifier step is swapped for `LogisticRegression`
and a new, model-appropriate parameter grid is searched over.

In [ ]:
# Replace RandomForestClassifier with LogisticRegression
pipeline.set_params(classifier=LogisticRegression(random_state=42))

# Point the grid search at the updated pipeline
grid_search.estimator = pipeline

# New parameter grid, specific to Logistic Regression
param_grid = {
    'classifier__solver': ['liblinear'],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__class_weight': [None, 'balanced']
}
grid_search.param_grid = param_grid

model = grid_search
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

### Classification report and confusion matrix for Logistic Regression

In [ ]:
print(classification_report(y_test, y_pred))

conf_matrix = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(conf_matrix, annot=True, cmap='Blues', fmt='d')
plt.title('Logistic Regression — Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## Model comparison and conclusion

- **Accuracy:** The two models perform almost the same overall — Random Forest scores
  around 84%, Logistic Regression around 83%.
- **True positive rate (recall on "Yes"):** Also nearly identical — both sit around
  50–51%, meaning each model correctly catches only about half of the actual rainy days.
- **Precision on "Yes":** Random Forest is notably more precise (~76%) than Logistic
  Regression (~69%) — when Random Forest predicts rain, it's right more often, i.e. fewer
  false alarms.
- **Overall:** Accuracy and recall are almost the same between the two models, but Random
  Forest is the better overall predictor once precision is factored in, since its rain
  predictions are more trustworthy. Neither model is a strongly reliable rainfall predictor
  on its own — both still miss roughly half of the days it actually rains, so which one to
  prefer in a real application depends on whether false alarms or missed rain days are more
  costly to get wrong.

**Possible next steps:** try feature engineering (e.g. rolling weather trends), test a
probability threshold other than 0.5 to trade precision for recall, try class-balancing
techniques (oversampling, `class_weight='balanced'` more broadly), or bring in more
locations/years of data.
